# Clasificación de monumentos antiguos

Esqueleto para entrenar y comparar los modelos de la actividad. Complete cada sección con su propio código.

| Modelo | Arquitectura | Pesos iniciales | Qué se entrena | Aumentación de datos |
|---|---|---|---|---|
| A | ResNet18 | Aleatorios (desde cero) | Toda la red | Sí |
| B | ResNet18 | ImageNet | Solo la capa final (red congelada) | Sí |
| C | ResNet18 | ImageNet | Toda la red (ajuste fino) | Sí |
| D | ResNet18 | Places365 | Solo la capa final (red congelada) | Sí |
| E | ResNet18 | Places365 | Toda la red (ajuste fino) | Sí |
| F | ResNet18 | Places365 | Toda la red (ajuste fino) | No |
| G | CNN propia (máximo un millón de parámetros) | Aleatorios (desde cero) | Toda la red | Sí |

## Datos

Las celdas de esta sección descargan el conjunto de datos, definen el preprocesamiento común a todos los modelos (224×224 píxeles y normalización con las estadísticas de ImageNet) y crean los cargadores de validación y prueba.

In [ ]:
import os
import copy
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms.v2 as transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

In [ ]:
import gdown

DATA_DIR = "ancient_monuments"
if not os.path.isdir(DATA_DIR):
    gdown.download(id="16w5icDoXwW932fXesjMTVlDvF-W4nQTd", output="ancient_monuments.zip", quiet=False)
    zipfile.ZipFile("ancient_monuments.zip").extractall(".")

CLASSES = sorted(os.listdir(f"{DATA_DIR}/train"))
for split in ["train", "valid", "test"]:
    print(split, {c: len(os.listdir(f"{DATA_DIR}/{split}/{c}")) for c in CLASSES})

In [ ]:
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # estadísticas de ImageNet

# Preprocesamiento determinista: validación, prueba y entrenamiento sin aumentación
eval_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(MEAN, STD),
])

def make_loader(split, transform, shuffle=False):
    data = datasets.ImageFolder(f"{DATA_DIR}/{split}", transform)
    return DataLoader(data, batch_size=32, shuffle=shuffle, num_workers=2)

valid_loader = make_loader("valid", eval_transform)
test_loader = make_loader("test", eval_transform)

Algunas fotografías de entrenamiento de cada clase:

In [ ]:
samples = datasets.ImageFolder(f"{DATA_DIR}/train").samples
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(12, 2.2 * len(CLASSES)))
for row, name in enumerate(CLASSES):
    paths = [path for path, label in samples if label == row][:5]
    for ax, path in zip(axes[row], paths):
        ax.imshow(plt.imread(path))
        ax.axis("off")
    axes[row, 0].set_title(name, loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## Aumentación de datos

Defina las transformaciones aleatorias para el conjunto de entrenamiento. Deben terminar con la misma conversión y normalización que `eval_transform`. Para el modelo sin aumentación, use `eval_transform`.

In [ ]:
augmentation = eval_transform   # TODO: sustituir por la aumentación del equipo

train_loader = make_loader("train", augmentation, shuffle=True)

## Modelo

Para los modelos A a F, construya una ResNet18 con los pesos iniciales del modelo, sustituya la capa final por una de cinco salidas y, si corresponde, congele el resto de la red. Para el modelo G, diseñe una red convolucional propia con un máximo de un millón de parámetros.

Los pesos de ImageNet se obtienen de `torchvision`. La celda siguiente descarga los de Places365; su archivo tiene 365 salidas y los nombres de sus parámetros llevan el prefijo `module.`.

In [ ]:
PLACES365_URL = "http://places2.csail.mit.edu/models_places365/resnet18_places365.pth.tar"
checkpoint = torch.hub.load_state_dict_from_url(PLACES365_URL, map_location="cpu")

# TODO

## Entrenamiento

Entrene 10 épocas. Guarde la exactitud de validación de cada época y conserve el modelo de la época con mayor exactitud de validación.

In [ ]:
# TODO

## Guardar y compartir el modelo

Guarde el `state_dict` del mejor modelo junto con la información necesaria para volver a usarlo (incluido el historial de validación) y compártalo con el equipo.

In [ ]:
# TODO

## Evaluación

Reúna los modelos guardados por el equipo, reconstruya la arquitectura de cada uno y cargue sus pesos. Compruebe que cada modelo reproduce su exactitud de validación y calcule su exactitud en `test`. Este conjunto se usa una sola vez, al final.

In [ ]:
# TODO

## Figura de resultados

Genere una sola imagen con dos paneles y el nombre del equipo en el título. A la izquierda, la exactitud de validación por época de los siete modelos. A la derecha, una gráfica de barras con su exactitud en el conjunto de prueba, con el valor sobre cada barra; la etiqueta del modelo G debe indicar su número de parámetros.

In [ ]:
# TODO